# Exploring the Relationship Between Solar Load Forecast and Meterology Features

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
repo_root = Path.home() / "Documents" / "Coding" / "ML_NYISOSolarForecast"

data_root = repo_root / "data"
processed_dir = data_root / "processed"
merged_out = processed_dir / "03_merged_data.csv"

In [3]:
df = pd.read_csv(merged_out, low_memory=False)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()

Shape: (509460, 12)
Columns: ['time_stamp', 'zone_name', 'actual_mw', 'forecast_mw', 'capacity_mw', 'time', 'temperature_2m', 'surface_pressure', 'cloud_cover', 'windspeed_10m', 'shortwave_radiation', 'year']


,time_stamp,zone_name,actual_mw,forecast_mw,capacity_mw,time,temperature_2m,surface_pressure,cloud_cover,windspeed_10m,shortwave_radiation,year
0,2020-11-17 05:00:00+00:00,CAPITL,0.0,0.0,NaN,2020-11-17T05:00,1.1,956.9,15,14.0,0.0,2020
1,2020-11-17 05:00:00+00:00,CENTRL,0.0,0.0,NaN,2020-11-17T05:00,1.1,956.9,15,14.0,0.0,2020
2,2020-11-17 05:00:00+00:00,DUNWOD,0.0,0.0,NaN,2020-11-17T05:00,1.1,956.9,15,14.0,0.0,2020
3,2020-11-17 05:00:00+00:00,GENESE,0.0,0.0,NaN,2020-11-17T05:00,1.1,956.9,15,14.0,0.0,2020
4,2020-11-17 05:00:00+00:00,HUD VL,0.0,0.0,NaN,2020-11-17T05:00,1.1,956.9,15,14.0,0.0,2020


In [4]:
df.columns = (
    df.columns.str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
    .str.replace("-", "_", regex=False)
)

df["time_stamp"] = pd.to_datetime(df["time_stamp"], utc=True, errors="coerce")

if "time" in df.columns:
    df["time"] = pd.to_datetime(df["time"], utc=True, errors="coerce")

numeric_cols = [
    "actual_mw",
    "forecast_mw",
    "capacity_mw",
    "temperature_2m",
    "surface_pressure",
    "cloud_cover",
    "windspeed_10m",
    "shortwave_radiation",
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

df["zone_name"] = df["zone_name"].astype(str).str.strip().str.upper()
df["time_local"] = df["time_stamp"].dt.tz_convert("America/New_York")
df["date_local"] = df["time_local"].dt.date
df["year_from_ts"] = df["time_local"].dt.year
df["quarter_local"] = df["time_local"].dt.quarter
df["month_local"] = df["time_local"].dt.month
df["day_local"] = df["time_local"].dt.day
df["dayofweek_local"] = df["time_local"].dt.dayofweek
df["hour_local"] = df["time_local"].dt.hour
df["is_weekend"] = df["dayofweek_local"].isin([5, 6]).astype(int)
df["is_daylight_proxy"] = (df["shortwave_radiation"] > 0).astype(int)

df["forecast_error_mw"] = df["actual_mw"] - df["forecast_mw"]
df["absolute_error_mw"] = (df["actual_mw"] - df["forecast_mw"]).abs()

if "time" in df.columns:
    same_time_mask = (df["time"] == df["time_stamp"]) | (df["time"].isna() & df["time_stamp"].isna())
    time_columns_identical = bool(same_time_mask.all())
    print("Time Column Like time_stamp:", time_columns_identical)

    if time_columns_identical:
        df = df.drop(columns=["time"])

print("Columns after the Check:", df.columns.tolist())
print(df.dtypes)

Time Column Like time_stamp: True
Columns after the Check: ['time_stamp', 'zone_name', 'actual_mw', 'forecast_mw', 'capacity_mw', 'temperature_2m', 'surface_pressure', 'cloud_cover', 'windspeed_10m', 'shortwave_radiation', 'year', 'time_local', 'date_local', 'year_from_ts', 'quarter_local', 'month_local', 'day_local', 'dayofweek_local', 'hour_local', 'is_weekend', 'is_daylight_proxy', 'forecast_error_mw', 'absolute_error_mw']
time_stamp                          datetime64[us, UTC]
zone_name                                           str
actual_mw                                       float64
forecast_mw                                     float64
capacity_mw                                     float64
temperature_2m                                  float64
surface_pressure                                float64
cloud_cover                                       int64
windspeed_10m                                   float64
shortwave_radiation                             float64
year      

In [5]:
df_system = df[df["zone_name"] == "SYSTEM"].copy()
df_zones = df[df["zone_name"] != "SYSTEM"].copy()
df_nyc = df[df["zone_name"] == "N.Y.C."].copy()

print("Full Shape:", df.shape)
print("System Only:", df_system.shape)
print("Zone Only:", df_zones.shape)
print("NYC Only:", df_nyc.shape)

Full Shape: (509460, 23)
System Only: (42455, 23)
Zone Only: (467005, 23)
NYC Only: (42455, 23)


In [6]:
schema_check = pd.DataFrame({
    "column": df.columns,
    "dtype": [str(df[c].dtype) for c in df.columns],
    "missing_count": [df[c].isna().sum() for c in df.columns],
    "missing_pct": [(df[c].isna().mean() * 100).round(6) for c in df.columns],
}).sort_values(["missing_pct", "column"], ascending=[False, True])

duplicate_summary = pd.DataFrame({
    "check": [
        "duplicate_full_rows_all",
        "duplicate_time_zone_all",
        "duplicate_timestamps_all",
        "duplicate_time_zone_zones_only",
        "duplicate_timestamps_system_only",
    ],
    "count": [
        df.duplicated().sum(),
        df.duplicated(subset=["time_stamp", "zone_name"]).sum(),
        df.duplicated(subset=["time_stamp"]).sum(),
        df_zones.duplicated(subset=["time_stamp", "zone_name"]).sum(),
        df_system.duplicated(subset=["time_stamp"]).sum(),
    ]
})

ts_min = df["time_stamp"].min()
ts_max = df["time_stamp"].max()

full_hours = pd.date_range(start=ts_min, end=ts_max, freq="h", tz="UTC")
observed_hours = pd.Index(df["time_stamp"].dropna().sort_values().unique())
missing_hours = full_hours.difference(observed_hours)

coverage_summary = pd.DataFrame({
    "expected_hour_count": [len(full_hours)],
    "observed_hour_count": [len(observed_hours)],
    "missing_hour_count": [len(missing_hours)],
})

missing_hours_df = pd.DataFrame({"missing_time_stamp": missing_hours})
missing_hours_df["time_local"] = missing_hours_df["missing_time_stamp"].dt.tz_convert("America/New_York")

schema_check, duplicate_summary, coverage_summary, missing_hours_df.head(20)

(                 column                             dtype  missing_count  \
 4           capacity_mw                           float64         286836   
 22    absolute_error_mw                           float64          12000   
 21    forecast_error_mw                           float64          12000   
 2             actual_mw                           float64           8544   
 3           forecast_mw                           float64           3456   
 7           cloud_cover                             int64              0   
 12           date_local                            object              0   
 16            day_local                             int32              0   
 17      dayofweek_local                             int32              0   
 18           hour_local                             int32              0   
 20    is_daylight_proxy                             int64              0   
 19           is_weekend                             int64              0   

In [7]:
zone_coverage = (
    df.groupby("zone_name", as_index=False)
      .agg(
          rows=("time_stamp", "size"),
          unique_hours=("time_stamp", "nunique"),
          actual_missing=("actual_mw", lambda s: s.isna().sum()),
          forecast_missing=("forecast_mw", lambda s: s.isna().sum()),
          capacity_missing=("capacity_mw", lambda s: s.isna().sum()),
          actual_missing_pct=("actual_mw", lambda s: round(s.isna().mean() * 100, 4)),
          forecast_missing_pct=("forecast_mw", lambda s: round(s.isna().mean() * 100, 4)),
          capacity_missing_pct=("capacity_mw", lambda s: round(s.isna().mean() * 100, 4)),
      )
      .sort_values("zone_name")
)

zone_coverage

,zone_name,rows,unique_hours,actual_missing,forecast_missing,capacity_missing,actual_missing_pct,forecast_missing_pct,capacity_missing_pct
0,CAPITL,42455,42455,712,288,23903,1.6771,0.6784,56.302
1,CENTRL,42455,42455,712,288,23903,1.6771,0.6784,56.302
2,DUNWOD,42455,42455,712,288,23903,1.6771,0.6784,56.302
3,GENESE,42455,42455,712,288,23903,1.6771,0.6784,56.302
4,HUD VL,42455,42455,712,288,23903,1.6771,0.6784,56.302
5,LONGIL,42455,42455,712,288,23903,1.6771,0.6784,56.302
6,MHK VL,42455,42455,712,288,23903,1.6771,0.6784,56.302
7,MILLWD,42455,42455,712,288,23903,1.6771,0.6784,56.302
8,N.Y.C.,42455,42455,712,288,23903,1.6771,0.6784,56.302
9,NORTH,42455,42455,712,288,23903,1.6771,0.6784,56.302


In [24]:
zone_sum_hourly = (
    df_zones.groupby("time_stamp", as_index=False)
    .agg(
        zone_actual_sum=("actual_mw", "sum"),
        zone_forecast_sum=("forecast_mw", "sum"),
        zone_count=("zone_name", "nunique"),
        zone_actual_nonmissing=("actual_mw", lambda s: s.notna().sum()),
        zone_forecast_nonmissing=("forecast_mw", lambda s: s.notna().sum()),
    )
)

system_hourly_base = df_system[["time_stamp", "actual_mw", "forecast_mw"]].rename(columns={
    "actual_mw": "system_actual_mw",
    "forecast_mw": "system_forecast_mw",
})

system_vs_zones = system_hourly_base.merge(zone_sum_hourly, on="time_stamp", how="left")

system_vs_zones["actual_diff_system_minus_zone_sum"] = (
    system_vs_zones["system_actual_mw"] - system_vs_zones["zone_actual_sum"]
)

system_vs_zones["forecast_diff_system_minus_zone_sum"] = (
    system_vs_zones["system_forecast_mw"] - system_vs_zones["zone_forecast_sum"]
)

system_vs_zones["complete_zone_actual_hour"] = system_vs_zones["zone_actual_nonmissing"] == 11
system_vs_zones["complete_zone_forecast_hour"] = system_vs_zones["zone_forecast_nonmissing"] == 11

system_vs_zone_summary = pd.DataFrame({
    "metric": [
        "system_hours",
        "hours_with_all_11_zone_actuals",
        "hours_with_all_11_zone_forecasts",
        "mean_actual_diff_system_minus_zone_sum",
        "median_actual_diff_system_minus_zone_sum",
        "max_abs_actual_diff_system_minus_zone_sum",
        "mean_forecast_diff_system_minus_zone_sum",
        "median_forecast_diff_system_minus_zone_sum",
        "max_abs_forecast_diff_system_minus_zone_sum",
    ],
    "value": [
        len(system_vs_zones),
        system_vs_zones["complete_zone_actual_hour"].sum(),
        system_vs_zones["complete_zone_forecast_hour"].sum(),
        system_vs_zones.loc[system_vs_zones["complete_zone_actual_hour"], "actual_diff_system_minus_zone_sum"].mean(),
        system_vs_zones.loc[system_vs_zones["complete_zone_actual_hour"], "actual_diff_system_minus_zone_sum"].median(),
        system_vs_zones.loc[system_vs_zones["complete_zone_actual_hour"], "actual_diff_system_minus_zone_sum"].abs().max(),
        system_vs_zones.loc[system_vs_zones["complete_zone_forecast_hour"], "forecast_diff_system_minus_zone_sum"].mean(),
        system_vs_zones.loc[system_vs_zones["complete_zone_forecast_hour"], "forecast_diff_system_minus_zone_sum"].median(),
        system_vs_zones.loc[system_vs_zones["complete_zone_forecast_hour"], "forecast_diff_system_minus_zone_sum"].abs().max(),
    ]
})

largest_system_actual_diffs = system_vs_zones.loc[
    system_vs_zones["complete_zone_actual_hour"]
].reindex(
    system_vs_zones.loc[system_vs_zones["complete_zone_actual_hour"], "actual_diff_system_minus_zone_sum"].abs().sort_values(ascending=False).index
)[[
    "time_stamp",
    "system_actual_mw",
    "zone_actual_sum",
    "actual_diff_system_minus_zone_sum",
    "zone_actual_nonmissing"
]]

largest_system_forecast_diffs = system_vs_zones.loc[
    system_vs_zones["complete_zone_forecast_hour"]
].reindex(
    system_vs_zones.loc[system_vs_zones["complete_zone_forecast_hour"], "forecast_diff_system_minus_zone_sum"].abs().sort_values(ascending=False).index
)[[
    "time_stamp",
    "system_forecast_mw",
    "zone_forecast_sum",
    "forecast_diff_system_minus_zone_sum",
    "zone_forecast_nonmissing"
]]

system_vs_zone_summary

,metric,value
0,system_hours,42455.000000
1,hours_with_all_11_zone_actuals,41743.000000
2,hours_with_all_11_zone_forecasts,42167.000000
3,mean_actual_diff_system_minus_zone_sum,0.006560
4,median_actual_diff_system_minus_zone_sum,0.000000
5,max_abs_actual_diff_system_minus_zone_sum,69.190000
6,mean_forecast_diff_system_minus_zone_sum,-0.006811
7,median_forecast_diff_system_minus_zone_sum,0.000000
8,max_abs_forecast_diff_system_minus_zone_sum,0.050000
